In [1]:
import os, sys
import shutil

import glob
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Qt5Agg')
from matplotlib import gridspec
import matplotlib.dates as mdates
plt.ion()
from datetime import datetime, timezone, timedelta
import timeit

from Toolshed import Classifier, Download, Toolbox, VegetationLine, Plotting, PlottingSeaborn, Transects

import mpl_toolkits as mpl
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset
from matplotlib.ticker import MaxNLocator

import seaborn as sns; sns.set()
import math
import geemap
import ee
import pprint
from shapely import geometry
from shapely.geometry import Point, LineString
import pandas as pd
import geopandas as gpd
import matplotlib.cm as cm
import pyproj
from IPython.display import clear_output
import scipy
from scipy import optimize
import csv
import math

# sklearn modules
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
import sklearn
if sklearn.__version__[:4] == '0.20':
    from sklearn.externals import joblib
else:
    import joblib

# coastsat modules
sys.path.insert(0, os.pardir)

ee.Initialize()
ee.Authenticate() # should only need to be run the first time after installation

os.chdir(os.path.dirname(os.path.abspath("train_new_classifier_water_PS.ipynb")))

In [22]:
# filepaths 
filepath_images = os.path.join(os.getcwd(), 'Data')
filepath_train = os.path.join(os.getcwd(), 'training_data')
filepath_models = os.path.join(os.getcwd(), 'Classification\\models')

In [3]:
# The points represent the corners of a bounding box that go around your site
sitename = 'imp_test'
# rivermouth_saltmarsh tideflats_island westernmouth_island river_imppond 
# Date range
dates = ['2023-01-01', '2025-01-01']
# Satellite missions
# Input a list of containing any/all of 'L5', 'L7', 'L8', 'L9', 'S2', 'PSScene4Band'
# L5: 1984-2013; L7: 1999-2017 (SLC error from 2003); L8: 2013-present; S2: 2014-present; L9: 2021-present
sat_list = ['PSScene4Band']
# Cloud threshold for screening out cloudy imagery (0.5 or 50% recommended)
cloud_thresh = 0.3
# Extract shoreline (wet-dry boundary) as well as veg edge
wetdry = True
# Directory where the data will be stored
filepath = Toolbox.CreateFileStructure(sitename, sat_list)

# Reference shoreline/veg line shapefile name (should be stored in a folder called referenceLines in Data)
# Line should be ONE CONTINUOUS linestring along the shore, stored as a shapefile in WGS84 coord system
referenceLineShp = sitename + '_ref.shp'
# Maximum amount in metres by which to buffer the reference line for capturing veg edges within
max_dist_ref = 150

# Return AOI from reference line bounding box and save AOI folium map HTML in sitename directory
referenceLinePath = os.path.join(filepath, 'referenceLines', referenceLineShp)
referenceLineDF = gpd.read_file(referenceLinePath)
polygon, point, lonmin, lonmax, latmin, latmax = Toolbox.AOIfromLine(referenceLinePath, max_dist_ref, sitename)
# It's recommended to convert the polygon to the smallest rectangle (sides parallel to coordinate axes)       
polygon = Toolbox.smallest_rectangle(polygon)

if len(dates)>2:
    daterange='no'
else:
    daterange='yes'
years = list(Toolbox.daterange(datetime.strptime(dates[0],'%Y-%m-%d'), datetime.strptime(dates[-1],'%Y-%m-%d')))

In [4]:
# put all the inputs into a dictionnary
inputs = {
    'polygon': polygon,
    'dates': dates, 
    'daterange':dates, 
    'sat_list': sat_list, 
    'sitename': sitename, 
    'filepath':filepath_images,
    'cloud_thresh': cloud_thresh
}

train_sites = [sitename]
print('Sites for training:\n%s\n'%train_sites)

Sites for training:
['imp_test']



In [5]:
inputs

{'polygon': [[[-66.2744605444048, 45.15592766832553],
   [-66.24830017132933, 45.15592766832553],
   [-66.24830017132933, 45.173241016871216],
   [-66.2744605444048, 45.173241016871216],
   [-66.2744605444048, 45.15592766832553]]],
 'dates': ['2023-01-01', '2025-01-01'],
 'daterange': ['2023-01-01', '2025-01-01'],
 'sat_list': ['PSScene4Band'],
 'sitename': 'imp_test',
 'filepath': 'c:\\MPA\\COASTGUARD\\COASTGUARD-1\\Data',
 'cloud_thresh': 0.3}

In [6]:
# inputs = Download.check_images_available(inputs)
Sat = Download.RetrieveImages(inputs, SLC=False)

retrieving image metadata...


In [7]:
Sat = [s for s in Sat if len(s) > 0]

In [8]:
metadata = {}

for i in range(len(inputs['sat_list'])):
    metadata[inputs['sat_list'][i]] = {
        'filenames': [], 
        'acc_georef': [], 
        'epsg': [], 
        'dates': []
    }

metadata = Download.CollectMetadata(inputs, Sat)

In [9]:
settings = {
    'filepath_train': filepath_train, # folder where the labelled images will be stored
    'labels':{'saltmarsh':1,'saturated sediment':2,'water':3,'veg':4}, # labels for the classifier,
    'colors':{'saltmarsh':[1, 0.65, 0],'saturated sediment':[1,0,1],'water':[0.1,0.1,0.7],'veg':[0.8,0.8,0.1]},
    'tolerance':0.01, # this is the pixel intensity tolerance, when using flood fill for sandy pixels
                             # set to 0 to select one pixel at a time
    'ref_epsg': 32619,
    'max_dist_ref': 500,
    # general parameters:
    'cloud_thresh': cloud_thresh,        # threshold on maximum cloud cover
    'output_epsg': 32619,     # epsg code of spatial reference system desired for the output   
    # quality control:
    'check_detection': True,    # if True, shows each shoreline detection to the user for validation
    'adjust_detection': True,  # if True, allows user to adjust the postion of each shoreline by changing the threhold
    'save_figure': True,        # if True, saves a figure showing the mapped shoreline for each image
    # [ONLY FOR ADVANCED USERS] shoreline detection parameters:
    'min_beach_area': 200,     # minimum area (in metres^2) for an object to be labelled as a beach
    'buffer_size': 250,         # radius (in metres) for buffer around sandy pixels considered in the shoreline detection
    'min_length_sl': 500,       # minimum length (in metres) of shoreline perimeter to be valid
    'cloud_mask_issue': True,  # switch this parameter to True if sand pixels are masked (in black) on many images  
    'sand_color': 'bright',    # 'default', 'dark' (for grey/black sand beaches) or 'bright' (for white sand beaches)
    # add the inputs defined previously
    'inputs': inputs,
    'projection_epsg': 32619,
    'hausdorff_threshold':3*(10**50)
}

In [10]:
settings

{'filepath_train': 'c:\\MPA\\COASTGUARD\\COASTGUARD-1\\training_data',
 'labels': {'saltmarsh': 1, 'saturated sediment': 2, 'water': 3, 'veg': 4},
 'colors': {'saltmarsh': [1, 0.65, 0],
  'saturated sediment': [1, 0, 1],
  'water': [0.1, 0.1, 0.7],
  'veg': [0.8, 0.8, 0.1]},
 'tolerance': 0.01,
 'ref_epsg': 32619,
 'max_dist_ref': 500,
 'cloud_thresh': 0.3,
 'output_epsg': 32619,
 'check_detection': True,
 'adjust_detection': True,
 'save_figure': True,
 'min_beach_area': 200,
 'buffer_size': 250,
 'min_length_sl': 500,
 'cloud_mask_issue': True,
 'sand_color': 'bright',
 'inputs': {'polygon': [[[-66.2744605444048, 45.15592766832553],
    [-66.24830017132933, 45.15592766832553],
    [-66.24830017132933, 45.173241016871216],
    [-66.2744605444048, 45.173241016871216],
    [-66.2744605444048, 45.15592766832553]]],
  'dates': ['2023-01-01', '2025-01-01'],
  'daterange': ['2023-01-01', '2025-01-01'],
  'sat_list': ['PSScene4Band'],
  'sitename': 'imp_test',
  'filepath': 'c:\\MPA\\COASTGU

In [11]:
for site in train_sites:
    settings['inputs']['sitename'] = site
    settings['cloud_mask_issue'] = False
    
    polygon = settings['inputs']['polygon']
    satname = 'PSScene4Band'  # or 'WV03' or other sensor name from your metadata
    
    Classifier.label_WV_images(metadata, polygon, satname, settings)

KeyboardInterrupt: 

In [12]:
train_sites

['imp_test']

In [13]:
# 
# A Multilayer Perceptron is trained with *scikit-learn*. To train the classifier, the training data needs to be loaded.
# 
# You can use the data that was labelled here and/or the original CoastSat training data.

# load labelled images
features,labelmaps = Classifier.load_labels(train_sites, settings)

Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of feature vectors: 16
Number of pixels per class in training data:
saltmarsh : 33802 pixels
saturated sediment : 21731 pixels
water : 122698 pixels
veg : 151000 pixels


In [14]:
#As the classes do not have the same number of pixels, it is good practice to subsample the very large classes 
n_samples = 80000

for key in ['saltmarsh','saturated sediment','water','veg']:
    n_total = features[key].shape[0]
    n_use = n_samples
    # Upsample with replacement if needed
    replace_flag = True if n_total < n_use else False
    features[key] = features[key][
        np.random.choice(n_total, n_use, replace=replace_flag), :
    ]

In [15]:
# print classes again
print('Re-sampled classifier features:')
for key in features.keys():
    print('%s : %d pixels'%(key,len(features[key])))

Re-sampled classifier features:
saltmarsh : 80000 pixels
saturated sediment : 80000 pixels
water : 80000 pixels
veg : 80000 pixels


In [16]:
# When the labelled data is ready, format it into X, a matrix of features, and y, a vector of labels:

# format into X (features) and y (labels) 
classes = ['saltmarsh','saturated sediment','water','veg']
labels = [1,2,3,0]
X,y = Classifier.format_training_data(features, classes, labels)

In [17]:
#train on 70% of the data and evaluate on the other 30%

# divide in train and test and evaluate the classifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=True, random_state=0)

In [18]:
runs = []
for i in range(5):
    start_time = timeit.default_timer()
    classifier = MLPClassifier(hidden_layer_sizes=(16,8,4), solver='adam')
    classifier.fit(X_train,y_train)
    runs.append(classifier.score(X_test,y_test))
    print('Accuracy: %0.4f' % classifier.score(X_test,y_test))
    print(str(round(timeit.default_timer() - start_time, 5)) + ' seconds elapsed')

Accuracy: 0.9867
114.97899 seconds elapsed
Accuracy: 0.9856
87.91256 seconds elapsed
Accuracy: 0.9830
46.15353 seconds elapsed
Accuracy: 0.9852
130.34503 seconds elapsed
Accuracy: 0.9858
106.36245 seconds elapsed


In [19]:
# cross-validation
scores = cross_val_score(classifier, X, y, cv=10)
print('Accuracy: %0.4f (+/- %0.4f)' % (scores.mean(), scores.std() * 2))
# Plot a confusion matrix:

Accuracy: 0.9114 (+/- 0.4409)


In [20]:
y_pred = classifier.predict(X_test)

Classifier.plot_confusion_matrix(y_test, y_pred,
                                    classes=['saltmarsh','saturated sediment','water','veg'],
                                    normalize=False);

# When satisfied with the accuracy and confusion matrix, train the model using ALL the training data and save it:

Confusion matrix, without normalization


In [23]:
classifier.fit(X,y)
# joblib.dump(classifier, os.path.join(filepath_models, sitename+'_MLPClassifier_Veg_S2.pkl'))
joblib.dump(classifier, os.path.join(filepath_models, 'NN_4classes_PS_habitats.pkl'))
# joblib.dump(classifier, os.path.join(filepath_models, 'MLPClassifier_Veg_Full.pkl'))
print(str(round(timeit.default_timer() - start_time, 5)) + ' seconds elapsed')

2891.31288 seconds elapsed


In [34]:
settings['inputs']['filepath'] = settings['filepath']


KeyError: 'filepath'

In [27]:
# Load a classifier that you have trained (specify the classifiers filename) and evaluate it on the satellite images.
# 
# This section will save the output of the classification for each site in a directory named \evaluation.

# load and evaluate a classifier
# get_ipython().run_line_magic('matplotlib', 'qt')
classifier = joblib.load(os.path.join(filepath_models, 'NN_4classes_PS_habitats.pkl'))
settings['output_epsg'] = 32619
settings['min_beach_area'] = 200
settings['buffer_size'] = 250
settings['min_length_sl'] = 500
settings['cloud_thresh'] = 0.5
settings['inputs'] = {}
# visualise the classified images
for site in train_sites:
    settings['inputs']['sitename'] = site[:site.find('.')] 
    # plot the classified images
    Classifier.evaluate_classifier(classifier, metadata, polygon, Sat, settings)

KeyError: 'filepath'

In [ ]:
settings['inputs']['sitename'] = 'DornochSummer'
sumfeatures = Classifier.load_labels(train_sites, settings)
settings['inputs']['sitename'] = 'DornochWinter'
winfeatures = Classifier.load_labels(train_sites, settings)

In [ ]:
n_samples = 10000
    
for key in ['veg', 'nonveg']:
    sumfeatures[key] =  sumfeatures[key][np.random.choice(sumfeatures[key].shape[0], n_samples, replace=False),:]
# print classes again
for key in sumfeatures.keys():
    print('%s : %d pixels'%(key,len(sumfeatures[key])))
for key in ['veg', 'nonveg']:
    winfeatures[key] =  winfeatures[key][np.random.choice(winfeatures[key].shape[0], n_samples, replace=False),:]
# print classes again
for key in winfeatures.keys():
    print('%s : %d pixels'%(key,len(winfeatures[key])))
        
classes = ['veg','nonveg']
labels = [1,2]
sumX,sumy = Classifier.format_training_data(sumfeatures, classes, labels)
winX,winy = Classifier.format_training_data(winfeatures, classes, labels)

In [ ]:
sumX_train, sumX_test, sumy_train, sumy_test = train_test_split(sumX, sumy, test_size=0.3, shuffle=True, random_state=0)
winX_train, winX_test, winy_train, winy_test = train_test_split(winX, winy, test_size=0.3, shuffle=True, random_state=0)

In [ ]:
classifier = MLPClassifier(hidden_layer_sizes=(100,50), solver='adam')
# train classifier on summer data
classifier.fit(sumX_train,sumy_train)
# test summer model on winter data
print('Summer model on winter data Accuracy: %0.4f' % classifier.score(winX_test,winy_test))

winy_pred = classifier.predict(winX_test)
Classifier.plot_confusion_matrix(winy_test, winy_pred,
                                    classes=['veg','nonveg'],
                                    normalize=False);

In [ ]:
classifier = MLPClassifier(hidden_layer_sizes=(100,50), solver='adam')
# train classifier on winter data
classifier.fit(winX_train,winy_train)
# test winter model on summer data
print('Winter model on summer data Accuracy: %0.4f' % classifier.score(sumX_test,sumy_test))

sumy_pred = classifier.predict(sumX_test)
Classifier.plot_confusion_matrix(sumy_test, sumy_pred,
                                    classes=['veg','nonveg'],
                                    normalize=False);